#FlashEats — Class 7 Challenge
Model the Business Workflow with Data
Client question
“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [ ]:
from pathlib import Path

for f in Path("/content").glob("*.zip"):
    print(f)


/content/flasheats-classroom-pack.zip


In [ ]:
import zipfile
from pathlib import Path

# Find ZIP files in /content
zip_files = list(Path("/content").rglob("*.zip"))

if not zip_files:
    raise FileNotFoundError(
        "No ZIP file found in /content. The file may not have been uploaded to this Colab runtime."
    )

print("Found ZIP:")
for z in zip_files:
    print("  ", z)

# Use the first ZIP
zip_path = zip_files[0]

# Extraction directory
extract_dir = Path("/content/flasheats_class7")
extract_dir.mkdir(parents=True, exist_ok=True)

print("\nExtracting:", zip_path)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extraction complete!")
print("Extracted to:", extract_dir)

Found ZIP:
   /content/flasheats-classroom-pack.zip

Extracting: /content/flasheats-classroom-pack.zip
Extraction complete!
Extracted to: /content/flasheats_class7


In [ ]:
db_files = list(extract_dir.rglob("flasheats.db"))

if not db_files:
    raise FileNotFoundError("flasheats.db was not found after extraction.")

for db in db_files:
    print("Database found:", db)

db = db_files[0]
BASE = db.parent.parent

print("\nBASE =", BASE)
print("DB   =", db)

Database found: /content/flasheats_class7/flasheats-classroom-pack/database/flasheats.db

BASE = /content/flasheats_class7/flasheats-classroom-pack
DB   = /content/flasheats_class7/flasheats-classroom-pack/database/flasheats.db


In [ ]:
import json
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

print("Using pack:", BASE)
print("Database:", BASE / "database" / "flasheats.db")

Using pack: /content/flasheats_class7/flasheats-classroom-pack
Database: /content/flasheats_class7/flasheats-classroom-pack/database/flasheats.db


In [ ]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])


orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


#Challenge 1 — Reconstruct the order lifecycle
Choose 3 orders:

one delivered on time,
one delivered late,
one with an intervention.
Build a timeline with:

event_time | event_type | actor | source_system

Include as many lifecycle events as the data supports.

Hint
Start from one order_id, collect events from each source, then sort by time.

In [ ]:
print("Variables containing FlashEats data:")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(name, obj.shape)

Variables containing FlashEats data:
orders (1603, 13)
customers (900, 5)
restaurants (60, 6)
drivers (120, 4)
tickets (202, 5)
customer_actions (2365, 6)
interventions (430, 6)
outcomes (1600, 6)


In [ ]:
for name in ["orders", "customer_actions", "interventions", "outcomes"]:
    df_temp = globals()[name]

    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    print("Shape:", df_temp.shape)
    print("Columns:", df_temp.columns.tolist())

    display(df_temp.head(3))


ORDERS
Shape: (1603, 13)
Columns: ['order_id', 'customer_id', 'restaurant_id', 'driver_id', 'city', 'created_at', 'promised_eta', 'pickup_at', 'actual_delivery_at', 'final_status', 'distance_km_estimate', 'traffic_bucket', 'weather_bucket']


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear



CUSTOMER_ACTIONS
Shape: (2365, 6)
Columns: ['action_id', 'order_id', 'customer_id', 'action_type', 'action_at', 'channel']


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app
2,ACT-00003,O01334,C0692,ETA_VIEWED,2026-08-01T17:54:00,mobile_app



INTERVENTIONS
Shape: (430, 6)
Columns: ['intervention_id', 'order_id', 'intervention_type', 'intervention_at', 'initiated_by', 'reason']


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution
2,INT-00003,O00496,DRIVER_REASSIGNMENT,2026-08-04T15:09:00,dispatch,slow_progress



OUTCOMES
Shape: (1600, 6)
Columns: ['order_id', 'final_status_norm', 'delivered_flag', 'late_flag', 'delay_min', 'outcome_bucket']


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late
2,O00003,delivered,1,0.0,-4.41,delivered_on_time


In [ ]:
# Check the raw values
orders[orders["order_id"].isin(["O00001", "O00003", "O00781"])][
    ["order_id", "created_at", "pickup_at", "actual_delivery_at"]
]

,order_id,created_at,pickup_at,actual_delivery_at
0,O00001,2026-08-13T12:56:00,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899
2,O00003,2026-08-01T19:35:00,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191
780,O00781,2026-08-22T12:33:00,2026-08-22T13:11:17.309762,2026-08-22T14:01:40.289785


In [ ]:
def build_order_timeline(order_id):
    events = []

    # =========================================================
    # ORDERS
    # =========================================================
    order = orders[orders["order_id"] == order_id]

    if not order.empty:
        row = order.iloc[0]

        timestamps = {
            "ORDER_CREATED": ("created_at", "customer"),
            "ORDER_PICKED_UP": (
                "pickup_at",
                f"driver:{row['driver_id']}"
            ),
            "ORDER_DELIVERED": (
                "actual_delivery_at",
                f"driver:{row['driver_id']}"
            )
        }

        for event_type, (column, actor) in timestamps.items():
            value = row[column]

            if pd.notna(value):
                parsed_time = pd.to_datetime(value, errors="coerce")

                if pd.notna(parsed_time):
                    events.append({
                        "event_time": parsed_time,
                        "event_type": event_type,
                        "actor": actor,
                        "source_system": "orders"
                    })

    # =========================================================
    # CUSTOMER ACTIONS
    # =========================================================
    actions = customer_actions[
        customer_actions["order_id"] == order_id
    ]

    for _, row in actions.iterrows():
        parsed_time = pd.to_datetime(
            row["action_at"],
            errors="coerce"
        )

        if pd.notna(parsed_time):
            events.append({
                "event_time": parsed_time,
                "event_type": row["action_type"],
                "actor": f"customer:{row['customer_id']}",
                "source_system": "customer_actions"
            })

    # =========================================================
    # INTERVENTIONS
    # =========================================================
    ints = interventions[
        interventions["order_id"] == order_id
    ]

    for _, row in ints.iterrows():
        parsed_time = pd.to_datetime(
            row["intervention_at"],
            errors="coerce"
        )

        if pd.notna(parsed_time):
            events.append({
                "event_time": parsed_time,
                "event_type": row["intervention_type"],
                "actor": row["initiated_by"],
                "source_system": "interventions"
            })

    # =========================================================
    # SORT
    # =========================================================
    timeline = pd.DataFrame(events)

    if timeline.empty:
        return timeline

    timeline = timeline.sort_values(
        "event_time"
    ).reset_index(drop=True)

    return timeline[
        ["event_time", "event_type", "actor", "source_system"]
    ]

In [ ]:
late_timeline = build_order_timeline("O00001")
on_time_timeline = build_order_timeline("O00003")
intervention_timeline = build_order_timeline("O00781")

print("===== LATE ORDER: O00001 =====")
display(late_timeline)

print("===== ON-TIME ORDER: O00003 =====")
display(on_time_timeline)

print("===== INTERVENTION ORDER: O00781 =====")
display(intervention_timeline)

===== LATE ORDER: O00001 =====


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,ORDER_CREATED,customer,orders
1,2026-08-13 13:08:00.000000,DRIVER_REASSIGNMENT,dispatch,interventions
2,2026-08-13 13:21:00.000000,ETA_VIEWED,customer:C0168,customer_actions
3,2026-08-13 13:35:49.825561,ORDER_PICKED_UP,driver:D103,orders
4,2026-08-13 14:20:36.000000,SUPPORT_OPENED,customer:C0168,customer_actions
5,2026-08-13 14:22:25.035899,ORDER_DELIVERED,driver:D103,orders


===== ON-TIME ORDER: O00003 =====


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer,orders
1,2026-08-01 19:57:50.820366,ORDER_PICKED_UP,driver:D039,orders
2,2026-08-01 20:13:28.061191,ORDER_DELIVERED,driver:D039,orders


===== INTERVENTION ORDER: O00781 =====


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,ORDER_CREATED,customer,orders
1,2026-08-22 12:54:00.000000,ETA_VIEWED,customer:C0103,customer_actions
2,2026-08-22 12:57:00.000000,PRIORITY_DISPATCH,operations,interventions
3,2026-08-22 13:11:17.309762,ORDER_PICKED_UP,driver:D103,orders
4,2026-08-22 14:01:40.289785,ORDER_DELIVERED,driver:D103,orders


In [ ]:
outcomes[outcomes["order_id"].isin(
    ["O00001", "O00003", "O00781"]
)][
    ["order_id", "final_status_norm",
     "delivered_flag", "late_flag",
     "delay_min", "outcome_bucket"]
]

,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
2,O00003,delivered,1,0.0,-4.41,delivered_on_time
780,O00781,delivered,1,1.0,13.07,delivered_late


#Challenge 2 — Define the canonical project model
Your model must support:

customer → orders
order → customer interactions
order → support interactions
order → interventions
order → outcome
For each table, document:

primary key,
important foreign keys,
grain.
Then explain why this model is better for the project than mirroring every source-system table.

In [ ]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


interventions (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


## Canonical Relationships

The project uses `order_id` as the central entity.

Customer → Orders → Customer Actions
                  ├── Support Tickets
                  ├── Interventions
                  └── Outcome

The relationships are:

- One customer can have many orders.
- One order can have many customer interactions.
- One order can have many support interactions.
- One order can have many interventions.
- One order has one final outcome.

---

## Table Documentation

| Table | Primary Key | Important Foreign Keys | Grain |
|---|---|---|---|
| `orders` | `order_id` | `customer_id`, `restaurant_id`, `driver_id` | One row per order |
| `customer_actions` | `action_id` | `order_id`, `customer_id` | One row per customer action |
| `support_tickets` | `ticket_id` | `order_id` | One row per support ticket |
| `interventions` | `intervention_id` | `order_id` | One row per intervention |
| `outcomes` | `order_id` | `order_id → orders.order_id` | One row per order outcome |

---

## 1. Orders

**Primary key:** `order_id`

**Important foreign keys:**
- `customer_id`
- `restaurant_id`
- `driver_id`

**Grain:** One row represents one order.

Important lifecycle fields include:
- `created_at`
- `promised_eta`
- `pickup_at`
- `actual_delivery_at`
- `final_status`

`orders` is the central entity because the main project KPI is the late-delivery rate.

---

## 2. Customer Actions

**Primary key:** `action_id`

**Important foreign keys:**
- `order_id`
- `customer_id`

**Grain:** One row represents one customer action associated with an order.

Examples include:
- `ETA_VIEWED`
- `SUPPORT_OPENED`

An order can have zero, one, or many customer actions.

Relationship:

`orders 1 → many customer_actions`

---

## 3. Support Tickets

**Primary key:** `ticket_id`

**Important foreign key:**
- `order_id`

**Grain:** One row represents one support ticket.

Examples of ticket categories include:
- `late_delivery`
- `eta_changed`
- `status_mismatch`
- `driver_not_moving`

An order can have zero, one, or many support tickets.

Some support tickets have a missing `order_id`, so those records cannot be safely attributed to a specific order without additional evidence.

---

## 4. Interventions

**Primary key:** `intervention_id`

**Important foreign key:**
- `order_id`

**Grain:** One row represents one operational intervention applied to an order.

Examples include:
- `PRIORITY_DISPATCH`
- `DRIVER_REASSIGNMENT`
- `CUSTOMER_CREDIT`

An order can have zero, one, or many interventions.

Relationship:

`orders 1 → many interventions`

---

## 5. Outcomes

**Primary key:** `order_id`

**Important foreign key:**
- `order_id → orders.order_id`

**Grain:** One row represents the final measured outcome of one order.

Important fields include:
- `delivered_flag`
- `late_flag`
- `delay_min`
- `outcome_bucket`

This provides the canonical project outcome used for the late-delivery KPI.

---

# Why This Canonical Model Is Better Than Mirroring Every Source-System Table

The project is trying to understand late delivery and the operational signals associated with it. An order-centric canonical model provides a consistent analytical structure across the different source systems.

It provides:

1. **Consistent relationships**  
   All order-level information can be connected through `order_id`.

2. **Clear grain**  
   Each table has a defined meaning:
   - `orders` → one order
   - `customer_actions` → one customer action
   - `support_tickets` → one support ticket
   - `interventions` → one intervention
   - `outcomes` → one final outcome

3. **Consistent KPI definition**  
   The `outcomes` table provides the canonical fields `late_flag`, `delay_min`, and `outcome_bucket`, reducing the risk of different parts of the analysis defining "late" differently.

4. **Lifecycle analysis**  
   Events from different sources can be connected to the same order to reconstruct its lifecycle from creation through delivery and final outcome.

5. **Reduced join ambiguity**  
   Keeping the grain explicit helps prevent accidental row multiplication when joining multiple one-to-many tables.

6. **Source-system independence**  
   The source tables can remain available for provenance, while the canonical model provides a stable structure for analysis.

Overall, the canonical model keeps the analysis centered on the order and its outcome while preserving customer interactions, support activity, and operational interventions as related event streams.

#Challenge 3 — Build interaction → intervention → outcome
Create one order-level table containing:

order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min

Answer:

How many late orders had support interaction?
How many orders received intervention?
Which intervention is most common?
Which frustrated journeys had no intervention?
Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [ ]:
# ============================================================
# Challenge 3 — Interaction → Intervention → Outcome
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Aggregate customer interactions to ONE ROW PER ORDER
# ------------------------------------------------------------

actions_by_order = (
    customer_actions
    .assign(
        action_type_upper=lambda x: x["action_type"].astype(str).str.upper()
    )
    .groupby("order_id")
    .agg(
        action_count=("action_id", "count"),
        support_opened=(
            "action_type_upper",
            lambda x: int((x == "SUPPORT_OPENED").any())
        ),
        cancel_attempted=(
            "action_type_upper",
            lambda x: int((x == "CANCEL_ATTEMPTED").any())
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 2. Aggregate interventions to ONE ROW PER ORDER
# ------------------------------------------------------------

interventions_by_order = (
    interventions
    .groupby("order_id")
    .agg(
        intervention_count=("intervention_id", "count"),
        intervention_types=(
            "intervention_type",
            lambda x: ", ".join(sorted(set(x.dropna().astype(str))))
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. Start with order-level information
# ------------------------------------------------------------

order_level = orders[
    ["order_id", "customer_id", "final_status"]
].copy()

# ------------------------------------------------------------
# 4. Join aggregated interactions and interventions
# ------------------------------------------------------------

order_level = order_level.merge(
    actions_by_order,
    on="order_id",
    how="left"
)

order_level = order_level.merge(
    interventions_by_order,
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# 5. Join canonical outcomes
# ------------------------------------------------------------

order_level = order_level.merge(
    outcomes[
        ["order_id", "late_flag", "delay_min"]
    ],
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# 6. Fill missing values
# ------------------------------------------------------------

order_level["support_opened"] = (
    order_level["support_opened"]
    .fillna(0)
    .astype(int)
)

order_level["cancel_attempted"] = (
    order_level["cancel_attempted"]
    .fillna(0)
    .astype(int)
)

order_level["intervention_count"] = (
    order_level["intervention_count"]
    .fillna(0)
    .astype(int)
)

order_level["intervention_types"] = (
    order_level["intervention_types"]
    .fillna("")
)

order_level["late_flag"] = (
    order_level["late_flag"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 7. Final required order-level table
# ------------------------------------------------------------

order_level_table = order_level[
    [
        "order_id",
        "customer_id",
        "support_opened",
        "cancel_attempted",
        "intervention_count",
        "intervention_types",
        "final_status",
        "late_flag",
        "delay_min"
    ]
].copy()

print("Final order-level table shape:", order_level_table.shape)

display(order_level_table.head(10))


# ============================================================
# Challenge Questions
# ============================================================

# 1. How many late orders had support interaction?
late_orders_with_support = order_level_table[
    (order_level_table["late_flag"] == 1) &
    (order_level_table["support_opened"] == 1)
]

print(
    "\n1. Late orders with support interaction:",
    len(late_orders_with_support)
)


# 2. How many orders received intervention?
orders_with_intervention = order_level_table[
    order_level_table["intervention_count"] > 0
]

print(
    "2. Orders that received intervention:",
    len(orders_with_intervention)
)


# 3. Which intervention is most common?
intervention_counts = (
    interventions["intervention_type"]
    .value_counts()
)

print("\n3. Intervention frequency:")
display(intervention_counts.to_frame("count"))

if len(intervention_counts) > 0:
    print(
        "Most common intervention:",
        intervention_counts.index[0],
        "(", intervention_counts.iloc[0], "orders/events )"
    )


# 4. Which frustrated journeys had no intervention?
#
# Define a frustrated journey as an order where the customer
# either opened support or attempted cancellation.
#
# Then filter to journeys with no intervention.

frustrated_no_intervention = order_level_table[
    (
        (order_level_table["support_opened"] == 1) |
        (order_level_table["cancel_attempted"] == 1)
    ) &
    (order_level_table["intervention_count"] == 0)
].copy()

print(
    "\n4. Frustrated journeys with no intervention:",
    len(frustrated_no_intervention)
)

display(
    frustrated_no_intervention[
        [
            "order_id",
            "customer_id",
            "support_opened",
            "cancel_attempted",
            "final_status",
            "late_flag",
            "delay_min"
        ]
    ].sort_values(
        ["late_flag", "delay_min"],
        ascending=[False, False]
    )
)

Final order-level table shape: (1603, 9)


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,1,0,1,DRIVER_REASSIGNMENT,delivered,1,10.82
1,O00002,C0043,0,0,0,,delivered,1,3.97
2,O00003,C0855,0,0,0,,delivered,0,-4.41
3,O00004,C0229,0,0,0,,cancelled,0,NaN
4,O00005,C0040,0,0,0,,delivered,1,21.02
5,O00006,C0152,0,0,0,,delivered,1,7.30
6,O00007,C0875,0,0,0,,delivered,1,4.27
7,O00008,C0223,1,0,1,CUSTOMER_CREDIT,delivered,1,26.86
8,O00009,C0685,0,0,0,,delivered,1,0.69
9,O00010,C0530,0,0,0,,delivered,0,-5.79



1. Late orders with support interaction: 138
2. Orders that received intervention: 430

3. Intervention frequency:


,count
intervention_type,
DRIVER_REASSIGNMENT,155
RESTAURANT_CONTACT,116
PRIORITY_DISPATCH,95
CUSTOMER_CREDIT,64


Most common intervention: DRIVER_REASSIGNMENT ( 155 orders/events )

4. Frustrated journeys with no intervention: 110


,order_id,customer_id,support_opened,cancel_attempted,final_status,late_flag,delay_min
736,O00737,C0824,1,0,delivered,1,36.28
210,O00211,C0629,1,0,delivered,1,32.38
976,O00977,C0254,1,0,delivered,1,31.92
994,O00995,C0746,1,0,delivered,1,31.72
892,O00893,C0556,1,0,delivered,1,30.32
...,...,...,...,...,...,...,...
1163,O01164,C0273,1,0,delivered,1,8.53
1086,O01087,C0714,1,0,delivered,1,8.45
234,O00235,C0200,1,0,delivered,1,8.44
476,O00477,C0117,1,0,delivered,1,8.37


## Challenge 3 — Findings

The order-level model shows that 138 late orders had a recorded support interaction, while 430 orders received at least one operational intervention.

Among the 430 intervention records, DRIVER_REASSIGNMENT was the most common intervention (155), followed by RESTAURANT_CONTACT (116), PRIORITY_DISPATCH (95), and CUSTOMER_CREDIT (64).

There were also 110 frustrated journeys with no recorded intervention, where frustration was defined as either SUPPORT_OPENED or CANCEL_ATTEMPTED. Several of these were substantially late delivered orders, indicating cases where customer frustration was recorded without a corresponding operational intervention.

These results describe associations in the operational data; they do not establish that support interactions caused delays or that interventions caused or prevented late delivery.

#Challenge 4 — Select 3–5 business metrics
Project KPI: Reduce Late Delivery Rate.

For each chosen metric, document:

metric name,
formula,
grain,
why it matters,
relationship to the project KPI.
At least one metric must represent:

an outcome,
a customer interaction,
an intervention.
# Example starting point:
# valid_outcomes=outcomes[outcomes.late_flag.notna()]
# late_delivery_rate=valid_outcomes.late_flag.mean()

# TODO: calculate your selected metrics

In [ ]:
# ============================================================
# Challenge 4 — Business Metrics
# ============================================================

# ------------------------------------------------------------
# 1. Outcome metric: Late Delivery Rate
# ------------------------------------------------------------

valid_outcomes = outcomes[
    outcomes["late_flag"].notna()
].copy()

late_delivery_rate = valid_outcomes["late_flag"].mean()

print(
    f"1. Late Delivery Rate: {late_delivery_rate:.2%}"
)


# ------------------------------------------------------------
# 2. Customer interaction metric: Support Interaction Rate
# ------------------------------------------------------------

support_interaction_rate = (
    order_level_table["support_opened"].mean()
)

print(
    f"2. Support Interaction Rate: {support_interaction_rate:.2%}"
)


# ------------------------------------------------------------
# 3. Intervention metric: Intervention Rate
# ------------------------------------------------------------

intervention_rate = (
    (order_level_table["intervention_count"] > 0).mean()
)

print(
    f"3. Intervention Rate: {intervention_rate:.2%}"
)


# ------------------------------------------------------------
# 4. Outcome severity metric: Average Delay Among Late Orders
# ------------------------------------------------------------

late_orders = outcomes[
    (outcomes["late_flag"] == 1) &
    (outcomes["delay_min"].notna())
]

avg_delay_late_orders = late_orders["delay_min"].mean()

print(
    f"4. Average Delay Among Late Orders: "
    f"{avg_delay_late_orders:.2f} minutes"
)

1. Late Delivery Rate: 56.39%
2. Support Interaction Rate: 8.61%
3. Intervention Rate: 26.82%
4. Average Delay Among Late Orders: 10.52 minutes


# Challenge 4 — Business Metrics

## Project KPI

**Project KPI: Reduce Late Delivery Rate**

The primary KPI is the percentage of valid orders that are delivered later than the promised ETA.

---

## Selected Metrics

| Metric | Formula | Grain | Why it matters | Relationship to Project KPI |
|---|---|---|---|---|
| **Late Delivery Rate** | `Late orders / Valid outcome orders` | Order | Directly measures the size of the late-delivery problem. | **Primary KPI.** Reducing this metric directly means fewer orders are delivered late. |
| **Support Interaction Rate** | `Orders with SUPPORT_OPENED / Total orders` | Order | Measures customer friction during the order journey. | Provides a customer-experience signal that can be analyzed alongside late delivery. |
| **Intervention Rate** | `Orders with ≥1 intervention / Total orders` | Order | Measures how frequently operations take an explicit action on an order. | Helps evaluate operational response to potentially problematic orders. |
| **Average Delay Among Late Orders** | `Total delay minutes among late orders / Number of late orders` | Late order | Measures the severity of delays rather than only their frequency. | Provides additional context on whether late deliveries are becoming less severe. |

---

## Metric 1 — Late Delivery Rate

**Measured value: 56.39%**

**Formula:**

`Late Delivery Rate = Late orders / Valid outcome orders`

**Grain:** Order

**Why it matters:**

This is the project's primary outcome metric. It measures how frequently orders are delivered later than the promised ETA.

**Relationship to KPI:**

This metric is the KPI itself. The project objective is to reduce it.

---

## Metric 2 — Support Interaction Rate

**Measured value: 8.61%**

**Formula:**

`Support Interaction Rate = Orders with SUPPORT_OPENED / Total orders`

**Grain:** Order

**Why it matters:**

A support interaction can represent customer friction during the order journey. Tracking this metric helps identify how frequently customers contact support.

**Relationship to KPI:**

Support interaction provides a customer-experience signal that can be compared with late-delivery outcomes. However, the metric does not establish that support interaction causes late delivery.

---

## Metric 3 — Intervention Rate

**Measured value: 26.82%**

**Formula:**

`Intervention Rate = Orders with at least one intervention / Total orders`

**Grain:** Order

**Why it matters:**

This measures how frequently operations take an explicit action on an order, such as driver reassignment, restaurant contact, priority dispatch, or customer credit.

**Relationship to KPI:**

This is an operational-response metric. Comparing intervention activity with late-delivery outcomes can help identify how interventions are being used around problematic orders.

The metric should not be interpreted as proof that interventions cause or prevent late delivery because interventions may be concentrated on orders that are already at higher risk.

---

## Metric 4 — Average Delay Among Late Orders

**Measured value: 10.52 minutes**

**Formula:**

`Average Delay = Total delay minutes among late orders / Number of late orders`

**Grain:** Late order

**Why it matters:**

Late Delivery Rate tells us **how frequently orders are late**, while Average Delay tells us **how severe the delays are**.

Using both metrics prevents us from looking only at the number of late orders while ignoring the magnitude of the delays.

**Relationship to KPI:**

This is a supporting outcome metric. It helps determine whether the severity of late deliveries is changing alongside the primary late-delivery rate.

---

## Current Metric Snapshot

| Metric | Current Value |
|---|---:|
| Late Delivery Rate | **56.39%** |
| Support Interaction Rate | **8.61%** |
| Intervention Rate | **26.82%** |
| Average Delay Among Late Orders | **10.52 min** |

## Interpretation

The current baseline shows that **56.39% of valid orders are classified as late**. Customer support interaction is recorded for **8.61% of orders**, while **26.82% of orders receive at least one intervention**. Among late orders, the average delay is **10.52 minutes**.

These metrics provide complementary views of the problem: the KPI measures the delivery outcome, support interaction measures customer friction, intervention rate measures operational response, and average delay measures the severity of late deliveries.

#Challenge 5 — Investigate the workflow with joins and aggregations
Answer at least three:

A. Do orders with support interactions have higher delay?
B. What is late rate with vs without intervention?
C. Which intervention type is associated with the lowest late rate?
D. Which restaurants contribute the largest number of late orders?
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

What does this tell the business, and what does it NOT prove?

# TODO: use the order-level model plus joins/groupby
# Reminder: association != causation


In [ ]:
# ============================================================
# Challenge 5 — Workflow Investigation
# ============================================================

# ------------------------------------------------------------
# A. Do orders with support interactions have higher delay?
# ------------------------------------------------------------

support_delay_analysis = (
    order_level_table[
        order_level_table["delay_min"].notna()
    ]
    .groupby("support_opened")
    .agg(
        orders=("order_id", "count"),
        avg_delay_min=("delay_min", "mean"),
        late_rate=("late_flag", "mean")
    )
    .reset_index()
)

support_delay_analysis["late_rate"] *= 100

print("A. Delay and late rate by support interaction")
display(support_delay_analysis)


# ------------------------------------------------------------
# B. Late rate with vs without intervention
# ------------------------------------------------------------

order_level_table["has_intervention"] = (
    order_level_table["intervention_count"] > 0
).astype(int)

intervention_late_analysis = (
    order_level_table
    .groupby("has_intervention")
    .agg(
        orders=("order_id", "count"),
        late_orders=("late_flag", "sum"),
        late_rate=("late_flag", "mean")
    )
    .reset_index()
)

intervention_late_analysis["late_rate"] *= 100

print("\nB. Late rate with vs without intervention")
display(intervention_late_analysis)


# ------------------------------------------------------------
# C. Which intervention type is associated with lowest late rate?
# ------------------------------------------------------------

# Join intervention records to canonical outcomes.
# One order can have multiple intervention records, so first
# calculate order-level late outcome for each intervention type.

intervention_outcome = interventions[
    ["order_id", "intervention_type"]
].merge(
    outcomes[
        ["order_id", "late_flag"]
    ],
    on="order_id",
    how="left"
)

intervention_type_analysis = (
    intervention_outcome
    .groupby("intervention_type")
    .agg(
        intervention_records=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        late_orders=("late_flag", "sum"),
        late_rate=("late_flag", "mean")
    )
    .reset_index()
)

intervention_type_analysis["late_rate"] *= 100

intervention_type_analysis = intervention_type_analysis.sort_values(
    "late_rate"
)

print("\nC. Late rate by intervention type")
display(intervention_type_analysis)


# ------------------------------------------------------------
# E. Support interaction + intervention + still late
# ------------------------------------------------------------

support_intervention_late = order_level_table[
    (order_level_table["support_opened"] == 1) &
    (order_level_table["intervention_count"] > 0) &
    (order_level_table["late_flag"] == 1)
].copy()

print(
    "\nE. Journeys with support interaction + intervention + still late:",
    len(support_intervention_late)
)

display(
    support_intervention_late[
        [
            "order_id",
            "customer_id",
            "support_opened",
            "intervention_count",
            "intervention_types",
            "final_status",
            "late_flag",
            "delay_min"
        ]
    ]
    .sort_values("delay_min", ascending=False)
)

A. Delay and late rate by support interaction


,support_opened,orders,avg_delay_min,late_rate
0,0,1360,1.732441,51.911765
1,1,138,16.675000,100.000000



B. Late rate with vs without intervention


,has_intervention,orders,late_orders,late_rate
0,0,1173,616,52.514919
1,1,430,228,53.023256



C. Late rate by intervention type


,intervention_type,intervention_records,unique_orders,late_orders,late_rate
2,PRIORITY_DISPATCH,95,95,44.0,51.162791
3,RESTAURANT_CONTACT,116,116,57.0,52.777778
1,DRIVER_REASSIGNMENT,155,155,85.0,57.823129
0,CUSTOMER_CREDIT,64,64,42.0,66.666667



E. Journeys with support interaction + intervention + still late: 32


,order_id,customer_id,support_opened,intervention_count,intervention_types,final_status,late_flag,delay_min
100,O00101,C0538,1,1,CUSTOMER_CREDIT,delivered,1,85.51
7,O00008,C0223,1,1,CUSTOMER_CREDIT,delivered,1,26.86
301,O00302,C0800,1,1,PRIORITY_DISPATCH,delivered,1,26.32
861,O00862,C0067,1,1,DRIVER_REASSIGNMENT,delivered,1,24.10
1577,O01578,C0774,1,1,PRIORITY_DISPATCH,delivered,1,22.76
953,O00954,C0217,1,1,DRIVER_REASSIGNMENT,delivered,1,21.62
818,O00819,C0342,1,1,DRIVER_REASSIGNMENT,delivered,1,20.04
883,O00884,C0177,1,1,DRIVER_REASSIGNMENT,delivered,1,19.59
287,O00288,C0760,1,1,RESTAURANT_CONTACT,delivered,1,19.09
1159,O01160,C0498,1,1,CUSTOMER_CREDIT,delivered,1,19.07


In [ ]:
# ============================================================
# D. Which restaurants contribute the largest number of late orders?
# ============================================================

restaurant_late_orders = (
    orders[
        ["order_id", "restaurant_id"]
    ]
    .merge(
        outcomes[
            ["order_id", "late_flag"]
        ],
        on="order_id",
        how="left"
    )
)

restaurant_late_orders = (
    restaurant_late_orders[
        restaurant_late_orders["late_flag"] == 1
    ]
    .groupby("restaurant_id")
    .agg(
        late_orders=("order_id", "count")
    )
    .reset_index()
    .sort_values("late_orders", ascending=False)
)

print("D. Restaurants contributing the largest number of late orders:")
display(restaurant_late_orders.head(10))

D. Restaurants contributing the largest number of late orders:


,restaurant_id,late_orders
49,R050,22
3,R004,20
23,R024,20
22,R023,20
29,R030,20
50,R051,20
44,R045,19
10,R011,19
24,R025,18
17,R018,18


# Challenge 5 — Workflow Investigation

## A. Do orders with support interactions have higher delay?

| Support interaction | Orders | Average delay (min) | Late rate |
|---|---:|---:|---:|
| No | 1,360 | 1.73 | 51.91% |
| Yes | 138 | 16.68 | 100.00% |

Orders with a recorded `SUPPORT_OPENED` interaction had an average delay of **16.68 minutes**, compared with **1.73 minutes** for orders without a support interaction.

The late rate was **100%** among orders with a support interaction, compared with **51.91%** among orders without one.

**What this tells the business:** Support interaction is strongly associated with problematic delivery journeys and can be used as a customer-friction signal or indicator of orders that require attention.

**What it does NOT prove:** It does not prove that contacting support causes the delay. Customers may open support because the order is already late or showing signs of a problem.

---

## B. What is the late rate with vs without intervention?

| Intervention | Orders | Late orders | Late rate |
|---|---:|---:|---:|
| No intervention | 1,173 | 616 | 52.51% |
| At least one intervention | 430 | 228 | 53.02% |

Orders with at least one intervention had a late rate of **53.02%**, compared with **52.51%** for orders without intervention.

The observed difference is approximately **0.51 percentage points**.

**What this tells the business:** In this dataset, intervention and non-intervention orders have very similar observed late rates. This suggests that intervention status alone is not sufficient to explain the final delivery outcome and should be examined together with other risk factors.

**What it does NOT prove:** It does not prove that interventions are ineffective. Interventions may be targeted toward orders that were already more likely to become late, creating selection/confounding effects.

---

## C. Which intervention type is associated with the lowest late rate?

| Intervention type | Intervention records | Unique orders | Late orders | Late rate |
|---|---:|---:|---:|---:|
| `PRIORITY_DISPATCH` | 95 | 95 | 44 | **51.16%** |
| `RESTAURANT_CONTACT` | 116 | 116 | 57 | **52.78%** |
| `DRIVER_REASSIGNMENT` | 155 | 155 | 85 | **57.82%** |
| `CUSTOMER_CREDIT` | 64 | 64 | 42 | **66.67%** |

`PRIORITY_DISPATCH` had the lowest observed late rate among the intervention types, at **51.16%**.

**What this tells the business:** Priority dispatch is associated with the lowest observed late rate among the intervention categories in this dataset and may be worth investigating further as part of the operational workflow.

**What it does NOT prove:** This does not establish that priority dispatch is the most effective intervention. Different interventions may be applied to different types or severities of delivery problems.

---

## E. Which journeys show support interaction + intervention + still late?

There are **32 orders** where all three conditions occurred:

- `SUPPORT_OPENED = 1`
- `intervention_count > 0`
- `late_flag = 1`

Examples include:

| Order | Intervention | Delay (min) |
|---|---|---:|
| O00101 | CUSTOMER_CREDIT | 85.51 |
| O00008 | CUSTOMER_CREDIT | 26.86 |
| O00302 | PRIORITY_DISPATCH | 26.32 |
| O00862 | DRIVER_REASSIGNMENT | 24.10 |
| O01578 | PRIORITY_DISPATCH | 22.76 |
| O00954 | DRIVER_REASSIGNMENT | 21.62 |
| O00819 | DRIVER_REASSIGNMENT | 20.04 |
| O00884 | DRIVER_REASSIGNMENT | 19.59 |

These journeys represent cases where **customer friction was recorded, an operational intervention occurred, and the order was still ultimately late**.

**What this tells the business:** These 32 journeys are a useful investigation cohort because they represent cases where both customer and operational signals were present but the final outcome remained late.

**What it does NOT prove:** It does not prove that the interventions failed. The orders may have been difficult cases where intervention reduced a potentially larger delay, and there is no counterfactual showing what would have happened without intervention.

---

# Overall Challenge 5 Finding

The strongest signal is the relationship between **support interaction and delivery delay**. Orders with support interaction had a **16.68-minute average delay and 100% late rate**, compared with **1.73 minutes and 51.91%** for orders without support interaction.

The intervention comparison is much less differentiated: orders with interventions had a **53.02% late rate**, compared with **52.51%** without intervention.

Among intervention types, `PRIORITY_DISPATCH` had the lowest observed late rate at **51.16%**, while `CUSTOMER_CREDIT` had the highest at **66.67%**.

The 32 journeys with **support interaction + intervention + late outcome** provide a focused cohort for deeper investigation.

These are observational associations. They should be used to identify where to investigate the workflow rather than interpreted as causal evidence of intervention effectiveness.

# Challenge 6 — Connect the model to the KPI
Create a one-page project view:

PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS

Answer:

Which metrics are directly controllable by operations?
Which are outcomes?
Which missing event limits the model most?
What would you instrument next?
Final deliverable:

core entities,
key events,
relationships,
3–5 metrics,
KPI linkage,
one modelling limitation.

# Challenge 6 — KPI-Linked Project View

### Project KPI

**Reduce late deliveries**

### Core entities

* **Orders** — canonical order-level lifecycle and delivery facts
* **Customers** — customer context
* **Restaurants** — restaurant context
* **Drivers** — delivery execution context
* **Customer Actions** — customer interactions such as ETA views and support
* **Support Tickets** — customer-reported issues
* **Interventions** — operational actions taken on orders
* **Outcomes** — canonical order-level delivery outcomes

### Key workflow

**Order Created → Promised ETA → Pickup/Delivery Operations → Customer Action/Support → Intervention → Actual Delivery → Outcome**

### KPI linkage

**Project KPI → Outcome Metrics → Workflow/Driver Metrics → Interventions → Data Sources/Events**

* **Outcome metrics:** Late Delivery Rate, Average Delay Among Late Orders
* **Workflow/driver metrics:** Support Interaction Rate, Intervention Rate, restaurant late-order volume
* **Interventions:** Driver Reassignment, Restaurant Contact, Priority Dispatch, Customer Credit
* **Data sources:** Orders, Customer Actions, Support Tickets, Interventions, Outcomes

### Metrics

| Metric                          | Type                   | Purpose                                            |
| ------------------------------- | ---------------------- | -------------------------------------------------- |
| Late Delivery Rate              | Outcome                | Measures overall late-delivery problem             |
| Average Delay Among Late Orders | Outcome                | Measures severity of late deliveries               |
| Support Interaction Rate        | Workflow signal        | Measures customer friction                         |
| Intervention Rate               | Operational            | Measures frequency of operational response         |
| Restaurant Late-Order Volume    | Workflow/driver signal | Identifies where late-order volume is concentrated |

Observed project values:

* **Late Delivery Rate:** 56.39%
* **Support Interaction Rate:** 8.61%
* **Intervention Rate:** 26.82%
* **Average Delay Among Late Orders:** 10.52 minutes

### Directly controllable by operations

**Intervention Rate and intervention choice** are the most directly controllable operational metrics. Support interactions and late deliveries are observed signals/outcomes rather than directly controllable quantities.

### Main modelling limitation

The model does not capture the **full history of ETA changes** as timestamped events. A single promised ETA cannot show whether the original promise was unrealistic or whether the ETA was progressively changed as the order became delayed.

### Next instrumentation

Add an **ETA-change event table** containing:

`eta_event_id, order_id, event_at, previous_eta, new_eta, reason, initiated_by`

This would allow the team to connect ETA changes, customer friction, interventions, and eventual late delivery more precisely.


## Final Reflection

A strong solution does not produce the largest schema. It produces the **smallest useful model that explains the workflow and supports the project KPI**.

For FlashEats, the **order** is the central entity, with customer actions, support tickets, interventions, and outcomes connected to it. This order-centric model makes it possible to trace the journey from order creation to final delivery and connect operational activity with the late-delivery KPI.

The analysis also shows the importance of keeping **workflow signals separate from outcomes**. Metrics such as intervention rate and support interaction rate describe what happened during the journey, while late-delivery rate and delivery delay measure the final outcome.

The biggest modelling limitation is the lack of a **timestamped ETA-change history**. Instrumenting ETA changes would provide better visibility into how delivery expectations evolved and would help connect operational events, customer friction, and late outcomes more accurately.

Overall, the goal is not to capture every possible data point, but to build the **smallest reliable model that explains the business problem, supports the KPI, and identifies what data should be collected next**.
